# 01 — Internal research, training, and comparison

This is the single runnable narrative for the internal Defactify study: environment checks, dataset audit, controlled rasterisation, FFT exploration, baseline execution, neural training, and internal analysis. Run sections in order. Heavy neural commands remain explicitly gated; no cell invents metrics when an artifact is absent.

**Scope:** exploratory internal evidence only. The confirmatory external and robustness phase is in `02_external_validation_and_results.ipynb`.

## 1. Environment and reproducibility


Run this notebook before a controlled experiment. It records the software environment and checks the locked H1-N preprocessing contract. It contains no model-accuracy claim.

**Status discipline.** D0 denotes the completed legacy diagnostic controls that exposed a geometry/source confound. H1-N denotes the amended, source-normalised comparison. D0 metrics are not H1-N results, and no completed H1-N neural run is assumed by this notebook. A model is not selected for an interface until the separately locked external evaluation is complete.

In [ ]:
from pathlib import Path
import json
import platform
import sys

import numpy as np
import pandas as pd
import torch

from ai_image_detector.features import (
    CONTROLLED_IMAGE_SIZE,
    CONTROLLED_PREPROCESSING_PROTOCOL,
    preprocessing_metadata,
)
from ai_image_detector.reproducibility import environment_snapshot, get_device, save_json, seed_everything


def find_repository_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src' / 'ai_image_detector').is_dir():
            return candidate
    raise RuntimeError('Open Jupyter from this repository or one of its subdirectories.')


REPO = find_repository_root()
SEED = 7
seed_everything(SEED)
DEVICE = get_device()
H1N_PREPROCESSING = preprocessing_metadata(CONTROLLED_PREPROCESSING_PROTOCOL)

assert H1N_PREPROCESSING['image_size'] == CONTROLLED_IMAGE_SIZE == 128
assert H1N_PREPROCESSING['train_crop'] == 'seeded_random_square_crop'
assert H1N_PREPROCESSING['eval_crop'] == 'center_square_crop'
assert H1N_PREPROCESSING['neural_train_augmentation']['horizontal_flip']['probability'] == 0.5

snapshot = environment_snapshot() | {
    'seed': SEED,
    'selected_device': str(DEVICE),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'h1n_preprocessing': H1N_PREPROCESSING,
}
save_json(snapshot, REPO / 'artifacts/environment/environment.json')
print(f'Repository: {REPO}')
snapshot

In [ ]:
artifact_root = REPO / 'artifacts'
completed_h1n = []
for run_path in artifact_root.glob('*/run.json'):
    run = json.loads(run_path.read_text(encoding='utf-8'))
    protocol = run.get('preprocessing', {}).get('protocol')
    metrics_path = run_path.parent / 'internal_test_metrics.json'
    if protocol == CONTROLLED_PREPROCESSING_PROTOCOL and metrics_path.is_file():
        completed_h1n.append(run_path.parent.name)

study_status = {
    'D0_legacy_diagnostics': 'completed; retained only as a confound diagnostic',
    'H1_N_completed_internal_runs': sorted(completed_h1n),
    'H1_N_confirmatory_external_evaluation': 'locked and pending',
    'deployable_model': 'none until the frozen external evaluation is reported',
}
study_status

## Acceptance checks

A controlled run may proceed only when the device and Git revision are recorded, the H1-N metadata says `h1n_square_crop_128_v1` and `128 × 128`, and the grouped-manifest gate in Notebook 01 passes. For neural H1-N RGB/FFT training, the train crop is a seeded random square crop; neural validation and evaluation use the deterministic centre crop. The controlled radial-logistic baseline is intentionally different: it extracts deterministic centre-crop features on train, validation, and test, with no augmentation. On Apple Silicon, record the MPS availability; do not treat CPU/MPS choice as a performance result.

In [ ]:
assert torch.__version__, 'PyTorch is unavailable'
print(json.dumps(snapshot, indent=2))
print('MPS available:', torch.backends.mps.is_available())
print('No accuracy, calibration, or deployment claim is produced in this notebook.')

## 2. Data audit and leakage-resistant split


This notebook audits the grouped Defactify manifest before H1-N training. The official split had cross-split caption and perceptual-hash overlap, so the group-disjoint manifest is the internal experimental frame. The original internal test was inspected during D0; therefore any H1-N result on it is an **exploratory internal stress-test result**, not the confirmatory result.

In [ ]:
from pathlib import Path

import pandas as pd

from ai_image_detector.manifest import audit_summary, load_manifest, split_overlap_report

MANIFEST = REPO / 'data/processed/defactify_grouped/manifest.csv'
assert MANIFEST.exists(), 'Run prepare_defactify.py and make_grouped_split.py first.'
frame = load_manifest(MANIFEST, check_paths=True)
required_columns = {'label', 'split', 'generator', 'width', 'height', 'leakage_group', 'group_id', 'source_id', 'sha256', 'phash'}
missing_columns = required_columns.difference(frame.columns)
assert not missing_columns, f'Manifest misses controlled-protocol columns: {sorted(missing_columns)}'
frame.head()

In [ ]:
summary = audit_summary(frame)
for name, value in summary.items():
    print(f'\n--- {name} ---')
    display(value) if hasattr(value, 'style') else print(value)

for key in ('source_id', 'group_id', 'leakage_group', 'caption', 'sha256', 'phash'):
    if key in frame.columns:
        leaked = split_overlap_report(frame, key)
        print(f'{key}: {len(leaked)} records in a cross-split group')
        if len(leaked):
            display(leaked.head())

## Why D0 required an amendment

The prepared corpus has a class-correlated geometry/source channel: real photographs have varied rectangular dimensions, whereas synthetic images are square, and no exact `(width, height)` pair is shared across the two labels. D0's metadata-only control and direct rectangular-to-square resizing therefore showed that geometry can produce a high score without demonstrating image provenance. In particular, anisotropic resizing can create a class-correlated frequency pattern before an FFT is calculated.

D0 remains a useful *diagnostic* record. It is not an H1-N baseline, an architecture-selection result, or evidence that an individual image is AI-generated.

In [ ]:
geometry = (
    frame.assign(aspect_ratio=frame['width'] / frame['height'])
    .groupby(['label', 'generator'], dropna=False)
    .agg(
        images=('path', 'size'),
        unique_widths=('width', 'nunique'),
        unique_heights=('height', 'nunique'),
        median_aspect_ratio=('aspect_ratio', 'median'),
    )
    .reset_index()
)

real_dimensions = set(map(tuple, frame.loc[frame.label == 0, ['width', 'height']].to_numpy()))
fake_dimensions = set(map(tuple, frame.loc[frame.label == 1, ['width', 'height']].to_numpy()))
geometry_gate = {
    'exact_width_height_pairs_shared_between_labels': len(real_dimensions & fake_dimensions),
    'real_images_square_fraction': float((frame.loc[frame.label == 0, 'width'] == frame.loc[frame.label == 0, 'height']).mean()),
    'fake_images_square_fraction': float((frame.loc[frame.label == 1, 'width'] == frame.loc[frame.label == 1, 'height']).mean()),
}
display(geometry)
geometry_gate

## H1-N paired group sampler

The neural H1-N RGB/FFT runs train from the group-disjoint manifest with `paired_group_balanced_v1`. Each group visit emits one real image and one fake sibling; each neural training image then receives its seeded random square crop and train-only horizontal flip. The crop/flip RNG is keyed by seed, epoch, and stable `source_id`, not the absolute checkout path. The fake generator is assigned from a balanced, seeded cycle, so large duplicate groups and a more frequent generator cannot obtain extra training weight. Neural validation/test use the deterministic centre crop. The controlled radial-logistic baseline does not use this sampler or random crop: it extracts one deterministic centre-crop feature vector per image on every split. This is a training balance mechanism; it does not make the internal test confirmatory.

In [ ]:
from itertools import islice

from ai_image_detector.training import (
    PAIRED_GROUP_BALANCED_SAMPLER,
    PairedGroupSampler,
)

train_frame = frame.loc[frame.split == 'train'].reset_index(drop=True)
sampler = PairedGroupSampler(train_frame, seed=7, group_column='leakage_group')
sampled_indices = list(islice(iter(sampler), 12))
sampled = train_frame.iloc[sampled_indices].copy()

for start in range(0, len(sampled_indices), 2):
    pair = sampled.iloc[start : start + 2]
    assert pair.label.tolist() == [0, 1]
    assert pair.leakage_group.nunique() == 1

display(sampled[['leakage_group', 'label', 'generator', 'path']])
assert sampler.metadata()['choice'] == PAIRED_GROUP_BALANCED_SAMPLER
sampler.metadata()

## Decision gate

Proceed only if all group/caption/near-duplicate checks are clean for the grouped manifest and the sampler check passes. H1-N then uses a source-normalised 128 × 128 raster for **both** representations; Section 3 of this notebook makes that contract visible. Do not use accuracy alone, D0 metrics, or a smoke run to choose a deployment model.

## 3. FFT representation exploration


This is descriptive analysis, not provenance proof. It visualises the exact H1-N pixel contract: decode to RGB, crop a source-coordinate square without padding, resize once to 128 × 128, then pass the same raster either to RGB or to the FFT-magnitude transform. It does **not** reproduce the legacy D0 direct-resize radial result.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from ai_image_detector.features import (
    CONTROLLED_IMAGE_SIZE,
    fft_magnitude,
    radial_power_spectrum,
    source_normalized_rasterize,
)
from ai_image_detector.manifest import load_manifest

frame = load_manifest(REPO / 'data/processed/defactify_grouped/manifest.csv', check_paths=True)
train = frame.loc[frame.split == 'train'].copy()

for group, candidate in train.groupby('leakage_group', sort=True):
    if {0, 1}.issubset(set(candidate.label)):
        chosen_group = group
        break
else:
    raise RuntimeError('No real/fake paired group was found in the grouped train split.')

examples = (
    train.loc[train.leakage_group == chosen_group]
    .sort_values(['label', 'generator'])
    .groupby('generator', as_index=False, group_keys=False)
    .head(1)
    .reset_index(drop=True)
)
examples[['leakage_group', 'label', 'generator', 'width', 'height', 'path']]

In [ ]:
fig, axes = plt.subplots(len(examples), 4, figsize=(16, 4 * len(examples)))
axes = np.atleast_2d(axes)

for row_axes, (_, row) in zip(axes, examples.iterrows(), strict=True):
    original = Image.open(row.path).convert('RGB')
    raster = source_normalized_rasterize(original, size=CONTROLLED_IMAGE_SIZE, train=False)
    magnitude = fft_magnitude(raster, size=CONTROLLED_IMAGE_SIZE)
    radial = radial_power_spectrum(raster, size=CONTROLLED_IMAGE_SIZE)

    assert raster.size == (CONTROLLED_IMAGE_SIZE, CONTROLLED_IMAGE_SIZE)
    row_axes[0].imshow(original)
    row_axes[0].set_title(f'original: {row.width}×{row.height}')
    row_axes[1].imshow(raster)
    row_axes[1].set_title('H1-N centre-crop then 128×128')
    row_axes[2].imshow(magnitude, cmap='magma')
    row_axes[2].set_title('FFT magnitude of common raster')
    row_axes[3].plot(radial)
    row_axes[3].set_title('radial spectrum of common raster')
    for axis in row_axes[:3]:
        axis.set_axis_off()

plt.tight_layout()

## What is controlled, and what is not

Neural H1-N RGB/FFT training uses a seeded random square crop; neural validation, exploratory internal test, robustness, and external evaluation use the deterministic centre crop shown above. The observed controlled radial-logistic baseline deliberately avoids stochastic feature extraction: it uses the deterministic centre crop on train, validation, and test, without augmentation. The radial curve in this static visualisation likewise comes from the deterministic evaluation raster. Letterboxing and direct rectangular-to-square resizing are prohibited because they expose geometry or interpolation as a possible label cue.

Visible spectral patterns remain hypothesis-generating. H1-N asks whether an FFT-magnitude ResNet-50 outperforms an equal-capacity RGB ResNet-50 under this shared rasterisation, across three predeclared seeds. A plot, a legacy radial score, or an individual softmax output does not establish an image's origin.

## 4. Controlled H1-N training and analysis


H1-N is a narrow representation comparison: FFT-magnitude ResNet-50 versus RGB ResNet-50, both trained from random initialisation, on the same source-normalised 128 × 128 raster and paired-group training stream. Neural training uses a seeded random square crop; neural validation and test use the deterministic centre crop. The controlled radial-logistic baseline is a separate deterministic feature baseline: it centre-crops all three splits and does not inherit neural train-time randomness or augmentation. The original direct-resize experiments are D0 diagnostics and are deliberately excluded from this comparison.

The six commands below are long-running. They are printed by default and run only when `RUN_H1N_NEURAL_TRAINING=1` is set in the notebook kernel environment. This prevents accidental heavy training while keeping the cells locally runnable.

In [ ]:
import json
import os
import shlex
import subprocess

import pandas as pd

from ai_image_detector.features import CONTROLLED_PREPROCESSING_PROTOCOL, preprocessing_metadata
from ai_image_detector.manifest import load_manifest

MANIFEST = REPO / 'data/processed/defactify_grouped/manifest.csv'
assert MANIFEST.is_file(), 'Run the grouped-split preparation before H1-N training.'
frame = load_manifest(MANIFEST, check_paths=True)
assert set(['train', 'val', 'test']).issubset(set(frame.split))
assert preprocessing_metadata(CONTROLLED_PREPROCESSING_PROTOCOL)['image_size'] == 128

print(frame.groupby(['split', 'label']).size().rename('n'))

## D0 record: read-only diagnostic evidence

The old radial-FFT and file-metadata artifacts may be inspected to document why the protocol changed. They must not enter a H1-N ranking, threshold choice, model card, or web interface. The observed controlled radial-logistic baseline is distinct from this legacy D0 radial artifact: it uses a deterministic H1-N centre crop for train, validation, and test and supplies one fixed pixel-only feature vector per image. The metadata control receives geometry/source information unavailable to the intended image model; its score is evidence of dataset bias, not detection quality.

In [ ]:
d0_rows = []
for run_name in ('radial_logistic_seed7', 'file_metadata_control_seed7'):
    metrics_path = REPO / 'artifacts' / run_name / 'internal_test_metrics.json'
    if metrics_path.is_file():
        d0_rows.append({'status': 'D0 diagnostic only', 'run': run_name, **json.loads(metrics_path.read_text())})

d0_table = pd.DataFrame(d0_rows)
if d0_table.empty:
    print('No D0 artifact was found. That is not an H1-N result.')
else:
    display(d0_table)

## Controlled radial baseline command

The observed radial logistic baseline is a deterministic fixed-feature control, not a neural run. Its command is printed by default and only runs when explicitly gated.

In [ ]:
def radial_baseline_command() -> list[str]:
    return [
        sys.executable,
        'scripts/run_baselines.py',
        '--manifest', str(MANIFEST.relative_to(REPO)),
        '--only', 'radial_fft_logistic',
        '--output-root', 'artifacts/h1n_controls',
        '--seed', '7',
        '--preprocessing-protocol', CONTROLLED_PREPROCESSING_PROTOCOL,
    ]

RADIAL_BASELINE_COMMAND = radial_baseline_command()
print(shlex.join(RADIAL_BASELINE_COMMAND))
if os.environ.get('RUN_H1N_RADIAL_BASELINE') == '1':
    subprocess.run(RADIAL_BASELINE_COMMAND, check=True, cwd=REPO)
else:
    print('PENDING: set RUN_H1N_RADIAL_BASELINE=1 only to run the deterministic radial baseline.')

## Predeclared H1-N neural runs

All six neural runs have equal epoch cap, batch size, optimizer family, early-stopping rule and validation-only threshold selection. `--from-scratch` is explicit: ImageNet pretraining is excluded because it is an RGB semantic prior without an equivalent FFT interpretation. The CLI default preprocessing is intentionally left unset in the command: its default is the locked `h1n_square_crop_128_v1` protocol, which selects the 128 × 128 raster, seeded random neural train crop, deterministic neural evaluation crop, and paired group sampler.

In [ ]:
SEEDS = (7, 17, 42)
REPRESENTATIONS = ('rgb', 'fft')

def experiment_dir(representation: str, seed: int) -> Path:
    return REPO / 'artifacts' / f'h1n_{representation}_resnet50_seed{seed}'

def train_command(representation: str, seed: int) -> list[str]:
    return [
        sys.executable,
        'scripts/run_experiment.py',
        '--manifest', 'data/processed/defactify_grouped/manifest.csv',
        '--representation', representation,
        '--output-dir', str(experiment_dir(representation, seed).relative_to(REPO)),
        '--seed', str(seed),
        '--epochs', '15',
        '--batch-size', '32',
        '--learning-rate', '0.0001',
        '--patience', '4',
        '--from-scratch',
    ]

TRAIN_COMMANDS = [train_command(representation, seed) for representation in REPRESENTATIONS for seed in SEEDS]
for command in TRAIN_COMMANDS:
    print(shlex.join(command))

if os.environ.get('RUN_H1N_NEURAL_TRAINING') == '1':
    for command in TRAIN_COMMANDS:
        subprocess.run(command, check=True, cwd=REPO)
else:
    print('PENDING: set RUN_H1N_NEURAL_TRAINING=1 only to launch the predeclared long runs.')

## Cluster-aware evaluation and paired comparison

After all six saved prediction files exist, analyse each run with leakage-group cluster bootstrap intervals. For every seed, the FFT command compares FFT minus RGB. The analysis reports per-generator slices, paired-group ranking accuracy and confidence intervals without selecting hyperparameters on the test rows. The original test has already been inspected in D0, so these are still exploratory internal stress-test analyses.

In [ ]:
def analysis_command(representation: str, seed: int) -> list[str]:
    command = [
        sys.executable,
        'scripts/analyze_predictions.py',
        '--experiment-dir', str(experiment_dir(representation, seed).relative_to(REPO)),
        '--bootstrap-repeats', '2000',
        '--seed', '20260829',
    ]
    if representation == 'fft':
        command.extend(['--compare-to', str(experiment_dir('rgb', seed).relative_to(REPO))])
    return command

ANALYSIS_COMMANDS = [analysis_command(representation, seed) for representation in REPRESENTATIONS for seed in SEEDS]
for command in ANALYSIS_COMMANDS:
    print(shlex.join(command))

if os.environ.get('RUN_H1N_ANALYSIS') == '1':
    for command in ANALYSIS_COMMANDS:
        subprocess.run(command, check=True, cwd=REPO)
else:
    print('PENDING: run only after all matched H1-N experiment directories are complete.')

## Three-seed aggregation

Aggregate only after all three predeclared seeds for one representation have complete analysis artifacts. The command passes every seed and never selects a best test-set run.

In [ ]:
def aggregate_command(representation: str) -> list[str]:
    experiment_args = [
        argument
        for seed in SEEDS
        for argument in ('--experiment-dir', str(experiment_dir(representation, seed).relative_to(REPO)))
    ]
    return [
        sys.executable,
        'scripts/aggregate_experiments.py',
        *experiment_args,
        '--output-dir', f'artifacts/h1n_{representation}_resnet50_aggregate',
    ]

for representation in REPRESENTATIONS:
    print(shlex.join(aggregate_command(representation)))

if os.environ.get('RUN_H1N_AGGREGATION') == '1':
    for representation in REPRESENTATIONS:
        subprocess.run(aggregate_command(representation), check=True, cwd=REPO)
else:
    print('PENDING: set RUN_H1N_AGGREGATION=1 only after all three analyses per representation complete.')

## H1-N decision rule

Report ROC-AUC, PR-AUC, balanced accuracy, macro-F1, class recalls, FPR at TPR 95%, per-generator values and paired group-ranking accuracy. FPR@TPR=95% is a descriptive value read from the evaluation-set ROC curve; it is not a validation-selected operating threshold. The classification threshold remains selected on validation only. Aggregate the three *predeclared* seeds; do not select a lucky seed. The only confirmatory evaluation is the locked Synthbuster + RAISE external corpus after the model family, preprocessing, seed aggregation, checkpoint rule and threshold rule have all been frozen.